## Part 2 - Data Mining

Clean and organize your dataset. You must explain:
    ● How you removed duplicates
    ● How you handled missing values
    ● How you identified AI-related vs. human-related contributions
    ● How you classified task types
    ● Any assumptions you made

You may classify contributions manually, automatically, or using a combination of both.

# 0. Data Loading

In [62]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

data = pd.read_csv("data/mined_pull_requests.csv")
print("Original dataset:")
print("Rows:", len(data))
print("Columns:", len(data.columns))


Original dataset:
Rows: 40214
Columns: 31


In [63]:
data.head()

,pr_id,pr_number,title,body,author,agent,is_ai,state,merged,created_at,...,n_files,additions,deletions,churn,repo_id,repo_full_name,repo_language,repo_stars,repo_forks,html_url
0,2876006908,3375,Improve list and collection materializers perf...,# Optimized Collection Materializers with Batc...,strickvl,Claude_Code,True,closed,False,2025-02-24 19:52:57+00:00,...,6.0,825.0,229.0,1054.0,314197645.0,zenml-io/zenml,Python,4768.0,529.0,https://github.com/zenml-io/zenml/pull/3375
1,2876533575,3651,Clarify error message for missing plugin versions,"When using buf generate, provide specific erro...",iainmcgin,Claude_Code,True,closed,False,2025-02-25 00:48:56+00:00,...,1.0,62.0,1.0,63.0,212465715.0,bufbuild/buf,Go,10115.0,316.0,https://github.com/bufbuild/buf/pull/3651
2,2882576893,158,docs: add CLAUDE.md for development guidance,Created a CLAUDE.md file containing build comm...,ashwin-ant,Claude_Code,True,closed,True,2025-02-26 19:15:09+00:00,...,1.0,33.0,0.0,33.0,867337374.0,modelcontextprotocol/inspector,TypeScript,5353.0,674.0,https://github.com/modelcontextprotocol/inspec...
3,2883083776,163,Add CLAUDE.md with SDK development guide,🤖 Generated with [Claude Code](https://docs.an...,ashwin-ant,Claude_Code,True,closed,True,2025-02-27 00:12:26+00:00,...,1.0,25.0,0.0,25.0,862578138.0,modelcontextprotocol/typescript-sdk,TypeScript,8910.0,1113.0,https://github.com/modelcontextprotocol/typesc...
4,2883609845,258,Upgrade to Go 1.24 and update dependencies,This PR updates the project to use Go 1.24 and...,umputun,Claude_Code,True,closed,True,2025-02-27 06:55:13+00:00,...,95.0,331256.0,312167.0,643423.0,630707714.0,umputun/spot,Go,451.0,42.0,https://github.com/umputun/spot/pull/258


# 1. Explore Data

In [64]:
print(data.columns)
print(data.describe())
print(data.isnull().sum())
print(data['agent'].unique())

Index(['pr_id', 'pr_number', 'title', 'body', 'author', 'agent', 'is_ai',
       'state', 'merged', 'created_at', 'closed_at', 'merged_at',
       'first_response_at', 'labels', 'n_labels', 'task_type', 'n_comments',
       'n_reviews', 'n_review_comments', 'n_reviewers', 'n_commits', 'n_files',
       'additions', 'deletions', 'churn', 'repo_id', 'repo_full_name',
       'repo_language', 'repo_stars', 'repo_forks', 'html_url'],
      dtype='str')
              pr_id      pr_number      n_labels    n_comments     n_reviews  \
count  4.021400e+04   40214.000000  40214.000000  40214.000000  40214.000000   
mean   3.035295e+09    7559.519272      0.678520      0.972845      0.718034   
std    2.804352e+08   23724.109953      0.966938      2.284179      2.558552   
min    2.256770e+09       1.000000      0.000000      0.000000      0.000000   
25%    3.070900e+09     487.000000      0.000000      0.000000      0.000000   
50%    3.146310e+09    1946.500000      1.000000      0.000000      

# 2. Clean Data
Dropping duplicates only if they occur in pr_id or html_rl as these are also pull numbers, sorting AI and Human so it's just AI instead of all names. 

In [65]:
df = data.copy()
AI_AGENTS = ["Claude_Code", "Copilot", "Cursor", "Devin", "OpenAI_Codex"]
df["contribution_type"] = np.where(df["agent"].isin(AI_AGENTS), "AI", "Human")
print(pd.crosstab(data["is_ai"], df["contribution_type"]))

print(df.head())


contribution_type     AI  Human
is_ai                          
False                  0   6618
True               33596      0
        pr_id  pr_number                                              title  \
0  2876006908       3375  Improve list and collection materializers perf...   
1  2876533575       3651  Clarify error message for missing plugin versions   
2  2882576893        158       docs: add CLAUDE.md for development guidance   
3  2883083776        163           Add CLAUDE.md with SDK development guide   
4  2883609845        258         Upgrade to Go 1.24 and update dependencies   

                                                body      author        agent  \
0  # Optimized Collection Materializers with Batc...    strickvl  Claude_Code   
1  When using buf generate, provide specific erro...   iainmcgin  Claude_Code   
2  Created a CLAUDE.md file containing build comm...  ashwin-ant  Claude_Code   
3  🤖 Generated with [Claude Code](https://docs.an...  ashwin-ant  Claude_

In [66]:
print("Duplicate PR IDs:", df["pr_id"].duplicated().sum()) 
print("Duplicate html_urls:", df["html_url"].duplicated().sum())

d = data[data["html_url"].duplicated(keep=False)]
print("Distinct agents per duplicated URL:")
print(d.groupby("html_url")["agent"].nunique().value_counts())


Duplicate PR IDs: 0
Duplicate html_urls: 56
Distinct agents per duplicated URL:
agent
2    56
Name: count, dtype: int64


In [67]:
print("Missing html_url:", df["html_url"].isna().sum())
n0 = len(df)
df = df.drop_duplicates(subset=["pr_id"]).drop_duplicates(subset=["html_url"])
print("Duplicates removed:", n0 - len(df))
print("Rows remaining:", len(df))
print(df["contribution_type"].value_counts())

Missing html_url: 0
Duplicates removed: 56
Rows remaining: 40158
contribution_type
AI       33566
Human     6592
Name: count, dtype: int64


In [68]:
cols = ["n_files", "additions", "deletions", "n_commits", "first_response_at"]
print(df.groupby("contribution_type")[cols].apply(lambda g: g.isna().mean()).round(3))


df["body"] = df["body"].fillna("")
df["labels"] = df["labels"].fillna("")
for c in ["n_labels", "n_comments", "n_reviews", "n_review_comments", "n_reviewers"]:
    df[c] = df[c].fillna(0)
    

                   n_files  additions  deletions  n_commits  first_response_at
contribution_type                                                             
AI                   0.015      0.015      0.015        0.0              0.604
Human                1.000      1.000      1.000        1.0              1.000


In [69]:
h = df[df["contribution_type"] == "Human"]
print(h[["n_comments", "n_reviews", "n_review_comments", "n_reviewers"]]
      .describe().loc[["mean", "max"]])

      n_comments  n_reviews  n_review_comments  n_reviewers
mean         0.0        0.0                0.0          0.0
max          0.0        0.0                0.0          0.0


In [70]:
print(df["state"].value_counts())
df["is_open"] = df["state"].eq("open")

state
closed    37390
open       2768
Name: count, dtype: int64


In [71]:
for c in ["created_at", "closed_at", "merged_at", "first_response_at"]:
    df[c] = pd.to_datetime(df[c])

In [72]:
print(df["contribution_type"].value_counts())
print("Agent breakdown:")
print(pd.crosstab(df["agent"], df["contribution_type"]))

contribution_type
AI       33566
Human     6592
Name: count, dtype: int64
Agent breakdown:
contribution_type     AI  Human
agent                          
Claude_Code          459      0
Copilot             4970      0
Cursor              1541      0
Devin               4827      0
Human                  0   6592
OpenAI_Codex       21769      0


In [73]:
df["merge_status"] = np.select(
    [df["merged"] == True, df["is_open"]],
    ["Merged", "Open"],
    default="Closed unmerged",
)

df["time_to_merge_hours"] = (df["merged_at"] - df["created_at"]).dt.total_seconds() / 3600
df["time_to_close_hours"] = (df["closed_at"] - df["created_at"]).dt.total_seconds() / 3600
df["time_to_first_response_hours"] = (df["first_response_at"] - df["created_at"]).dt.total_seconds() / 3600
df["change_size"] = df["additions"] + df["deletions"]

time_cols = ["time_to_merge_hours", "time_to_close_hours", "time_to_first_response_hours"]
for c in time_cols:
    print(c, "negative:", (df[c] < 0).sum())
    df.loc[df[c] < 0, c] = np.nan

time_to_merge_hours negative: 0
time_to_close_hours negative: 0
time_to_first_response_hours negative: 0


In [74]:
print(df["task_type"].value_counts(dropna=False))

df["task_category"] = df["task_type"].fillna("unknown").str.strip().str.lower()

#revert is rare, so group it with other to do easier analysis
df["task_group"] = df["task_category"].replace({"revert": "other"}) 

print("\nShare of each group's PRs by task (columns sum to 1):")
print(pd.crosstab(df["task_group"], df["contribution_type"], normalize="columns").round(3))

task_type
feat        16369
fix          9863
docs         4405
refactor     2653
test         2537
chore        1743
build        1241
ci            511
perf          428
style         246
other         146
revert         16
Name: count, dtype: int64

Share of each group's PRs by task (columns sum to 1):
contribution_type     AI  Human
task_group                     
build              0.019  0.093
chore              0.027  0.128
ci                 0.012  0.015
docs               0.116  0.079
feat               0.430  0.294
fix                0.241  0.268
other              0.001  0.017
perf               0.010  0.013
refactor           0.068  0.056
style              0.006  0.009
test               0.070  0.028


In [75]:
closed = df[~df["is_open"]].copy()
print("Closed PRs:", len(closed))
print("\nMerge rate by group (closed PRs only):")
print((closed["merged"] == True).groupby(closed["contribution_type"]).mean().round(3))

Closed PRs: 37390

Merge rate by group (closed PRs only):
contribution_type
AI       0.768
Human    0.826
Name: merged, dtype: float64


# 3. Final Dataset

In [76]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nAI vs Human:")
print(df["contribution_type"].value_counts())

print("\nTask groups:")
print(df["task_group"].value_counts())

print("\nMerge status:")
print(df["merge_status"].value_counts())

Rows: 40158
Columns: 40

Missing values:
time_to_first_response_hours    26855
first_response_at               26855
merged_at                       11099
time_to_merge_hours             11099
additions                        7083
deletions                        7083
churn                            7083
n_files                          7083
change_size                      7083
n_commits                        6608
closed_at                        2768
time_to_close_hours              2768
repo_language                     265
repo_id                            49
repo_full_name                     49
dtype: int64

AI vs Human:
contribution_type
AI       33566
Human     6592
Name: count, dtype: int64

Task groups:
task_group
feat        16369
fix          9863
docs         4405
refactor     2653
test         2537
chore        1743
build        1241
ci            511
perf          428
style         246
other         162
Name: count, dtype: int64

Merge status:
merge_status
Merged     

In [77]:
df.to_csv(
    "data/cleaned_pull_requests.csv",
    index=False
)

print("\nCleaned dataset saved to:")
print("../data/cleaned_pull_requests.csv")


Cleaned dataset saved to:
../data/cleaned_pull_requests.csv
